## 3.4 MAC 帧 FER 扫描与吞吐量分析

在上一节中，我们在单 SNR 下演示了三种帧类型的传输。本节在多个 SNR 点扫描三种帧的 FER，对比帧长对误帧率的影响，并分析数据帧的有效吞吐量。

本节学习大纲如下：

- 三种帧类型 FER 对比
- 数据帧吞吐量分析

---

### 1. 导入与设置

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.sim.link_sim import sim_mac_signaling_link, sim_mac_data_link, sim_mac_mux_link

snr_range = np.arange(0, 16, 2)
n_frames = 50

---

### 2. 三种帧类型 FER 对比

In [ ]:
sig = sim_mac_signaling_link(snr_range_db=snr_range, n_frames=n_frames, mcs_index=7)
data = sim_mac_data_link(payload_sizes=[10], snr_range_db=snr_range, n_frames=n_frames, mcs_index=7)
mux = sim_mac_mux_link(snr_range_db=snr_range, n_frames=n_frames, mcs_index=7)

sig_fer = [1.0 - r for r in sig["signaling_success_rate"]]
data_fer = data["results"][10]["fer"]
mux_fer = [1.0 - r for r in mux["mux_success_rate"]]

print(f"{'SNR':>5s}  {'Signaling':>10s}  {'Data':>10s}  {'Mux':>10s}")
for s, sf, df, mf in zip(snr_range, sig_fer, data_fer, mux_fer):
    print(f"{s:5.0f}  {sf:10.4f}  {df:10.4f}  {mf:10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(snr_range, [max(f, 1e-4) for f in sig_fer], "o-", label="Signaling")
ax.semilogy(snr_range, [max(f, 1e-4) for f in data_fer], "s-", label="Data (10B)")
ax.semilogy(snr_range, [max(f, 1e-4) for f in mux_fer], "^-", label="Mux")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Frame Error Rate")
ax.set_title("MAC Frame Type FER Comparison (MCS=7)")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5); ax.set_ylim(bottom=1e-4)
plt.show()

---

### 3. 数据帧吞吐量

有效吞吐量 TP = (1-FER) x 频谱效率 x 符号速率。MCS=7 对应 SE=1.75 bit/symbol，Rs=1 MHz。

In [ ]:
se = 1.75
sr = 1e6
tp_list = [(1.0 - f) * se * sr / 1000 for f in data_fer]

print(f"{'SNR':>5s}  {'FER':>10s}  {'TP (kbps)':>12s}")
for s, f, tp in zip(snr_range, data_fer, tp_list):
    print(f"{s:5.0f}  {f:10.4f}  {tp:12.1f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(snr_range, tp_list, "s-", label="Data Throughput")
ax.axhline(y=100, color="gray", ls="--", alpha=0.5, label="100 kbps")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Throughput (kbps)")
ax.set_title("Data Frame Throughput (MCS=7)")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

## 课后实践

请补全下方多载荷 FER 对比扫描中的 **3 处空缺**（每处一行代码），在 [4, 10, 27] 三种载荷下对比数据帧的 FER-SNR 曲线。

要求：

1. **FER 提取**：从结果中获取特定载荷大小的 FER 列表
2. **吞吐量计算**：TP = (1-FER) × 频谱效率 × 符号速率
3. **曲线绘制**：对每种载荷画出 FER 曲线

完成后运行  观察不同载荷下的吞吐量差异。

In [ ]:
%%writefile fer_payload_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.sim.link_sim import sim_mac_data_link

snr_range = np.arange(0, 16, 2)
payload_sizes = [4, 10, 27]
se = 1.75          # MCS=7 频谱效率 (bit/symbol)
sr = 1e6           # 符号速率 (Hz)

result = sim_mac_data_link(payload_sizes=payload_sizes,
    snr_range_db=snr_range, n_frames=50, mcs_index=7)

# ==== : 补全对比分析（3处空缺）====

fig, ax = plt.subplots(figsize=(8, 5))
for size in payload_sizes:
    #  1: 提取该载荷大小的 FER 列表
    fers = ______________
    #  2: 绘制 FER 曲线
    ______________

    # 吞吐量
    print(f"Payload={size}B:")
    for s, f in zip(snr_range, fers):
        #  3: 计算有效吞吐量 (kbps)
        tp = ______________
        if s % 4 == 0:
            print(f"  SNR={s:2d} dB: FER={f:.3f}, TP={tp:.1f} kbps")

ax.set_xlabel("SNR (dB)"); ax.set_ylabel("FER")
ax.set_title("Data Frame FER vs Payload Size")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5)
ax.set_ylim(bottom=1e-4); plt.show()


执行以下命令进行编译并验证结果：


In [ ]:
!python fer_payload_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/03.04_answer.txt
